# BP5 Gate 7 — Executive Rollup Report
**Customer360 Navigator Enterprise Suite — Root Cause & Driver Analytics**

## Why this gate exists, and why it looks different from BP3's/BP4's own Gate 7

Gate 7 is a required closing deliverable for every Business Problem's cycle — a project-wide
convention established with BP1/BP2 and continued (in structure, not content) by BP3 and BP4.
Where Gate 6 is the technical/engineering closing gate (model card, monitoring, governance,
structural pass/fail checks), Gate 7 is the audience-facing synthesis gate: it trains and
evaluates nothing itself, and instead reads every real artifact Gates 1–6 already produced and
re-packages it into four polished, stakeholder-consumable deliverables (an interactive HTML
dashboard, a Word report, an Excel workbook, a PowerPoint deck) plus a JSON audit-trail manifest.

Three real, deliberate adaptations from BP3's/BP4's own Gate 7 pattern, disclosed here rather
than silently diverging (full detail also lives in `src/reporting/bp5_rollup_helpers.py`'s own
module docstring):

1. **BP5 has TWO real outcomes**, never one — every table, figure, and KPI in this report is
   outcome-aware, shown side by side for `outcome_1_intervention_required` and
   `outcome_2_timely_response_failure` rather than toggled, so both outcomes are always visible
   for direct comparison.
2. **This gate builds on top of Gate 5's own synthesis, not on Gate 3's/Gate 4's raw artifacts a
   second time.** BP5's own Gate 5 already produced a synthesized, citation-backed "prioritized
   root-cause report" per outcome (field-level ranking, category-level findings, SHAP findings,
   a champion validation snapshot) — re-parsing Gate 3's raw CSVs here would be redundant
   re-derivation. BP3's and BP4's own Gate 5 outputs were per-complaint decision records / cluster
   rollups, not a synthesized report, so their Gate 7 had no equivalent shortcut available. Gate 4's
   real calibration curve (the one real number not already in Gate 5's snapshot) is the one
   artifact this gate reads directly.
3. **"Recommended for Decision-Support Use," never "Recommended for Production."** BP5 is never a
   deployed classifier or inference service (Gate 6's own disclosed scope) — its two per-outcome
   champions exist purely to generate SHAP-based association evidence, never a deployment
   artifact. The champion validation snapshot is presented as supporting context for the SHAP
   findings, never as a deployment-readiness signal.

## The 3-tier recommendation, and how its Tier 2 trigger differs from both BP3's and BP4's

ECOA/Reg B disparate-impact applicability is `NOT_APPLICABLE` to BP5 (Master Plan Section 9,
confirmed at every gate) — exactly like BP4. Unlike BP4 (where Tier 2 is therefore **structurally
unreachable**, because BP4 has no other real flaggable condition), **BP5 does have its own real,
non-ECOA governance signals that can trigger a genuine Tier 2** ("Recommended for Decision-Support
Use, With Monitoring"): Gate 6's own live open-item detection (negligible-association candidate
fields, near-random PR-AUC, near-zero precision at the 0.5 threshold). This real run's own numbers
land in Tier 2 (2 negligible-strength fields + 1 near-zero-precision outcome, live-detected) — a
genuine, disclosed monitoring signal, never a legal or compliance determination, and never silently
hidden or upgraded to Tier 1.

Tier 3 ("Not Recommended for Decision-Support Use") triggers only if any structural check fails:
Gate 6's own pytest/notebook-syntax results, Gate 4 genuinely reconfirming Gate 3 for both
outcomes, the barred-fields bar never being relaxed, and the UDAAP language check passing.

## What this gate does

1. **KPI / detail assembly** (Section 4) — `build_kpi_bundle`, `build_gate1_summary`,
   `build_gate6_governance_detail`, `build_smart_suggestions`, and outcome-aware field/category/SHAP
   DataFrames, each built once (HYPER) from Gate 5's own two report JSONs.
2. **Static figures rendered once** (Section 5) — 8 matplotlib PNGs (Cramér's V bar, SHAP bar,
   calibration curve, confusion-derived rates bar — × 2 outcomes), reused as raw bytes across DOCX
   and PPTX (never re-rendered per format).
3. **Interactive HTML dashboard** (Section 6) — plain string `.replace()` templating (not Jinja2,
   not f-strings, to avoid brace-escaping conflicts with the template's own CSS/JS), Plotly.js via
   CDN with a `defer` attribute and a graceful text-fallback if the CDN is unreachable (verified in
   this gate's own sandbox check — see below), a live-filterable/searchable category-level findings
   table, and the same Sora/navy visual identity BP1–BP4 already established.
4. **DOCX / XLSX / PPTX exports** (Sections 7–9) — via `write_docx_report` / `write_xlsx_workbook`
   / `write_pptx_deck`, directly in Python (python-docx / openpyxl / python-pptx), no separate
   conversion step.
5. **Manifest write** (Section 10) — a flat `executive_rollup_manifest.json`, written under
   `notebooks/bp5_root_cause_driver_analytics/artifacts/` (not inside `reports/`), recording output
   paths/sizes, the recommendation tier, and hardcoded `contains_financial_impact_section: False` /
   `contains_assumption_based_content: False` declarations.
6. **Structural integrity checks** (Section 11) — 18 named assertions: every export reopens
   cleanly, expected sheet/slide counts, no financial-content leakage (checked in both the XLSX
   sheet names and the dashboard's own embedded JSON/HTML), no unresolved `__PLACEHOLDER__` tokens,
   the dashboard's embedded JSON reparses cleanly and its row counts match the real source
   DataFrames exactly, and the recommendation tier code is valid.

## Two real bugs this gate's own sandbox verification caught and fixed before delivery

1. **The shared design-system `PALETTE`/`CATEGORICAL_SEQUENCE` constants were NOT actually reused
   verbatim, despite this module's own docstring claiming they were.** While building
   `src/reporting/bp5_rollup_helpers.py`, the palette was written from memory rather than copied
   from BP3's/BP4's own real `bp{3,4}_rollup_helpers.py` files — the key names (`navy`/`accent`/…)
   and hex values did not match the real, established cross-BP design system (`primary_navy`
   `#1E2761`, `accent_blue` `#4C6EF5`, etc., confirmed identical in both BP3's and BP4's own real
   modules) at all. This would have shipped a genuinely different color identity for BP5's rollup
   than every other BP's, directly contradicting the "one visual identity across every BP" standing
   rule this project has followed since BP1. **Fixed** by reading BP3's and BP4's real
   `bp{3,4}_rollup_helpers.py` files directly and replacing the fabricated palette with their exact
   key names and hex values, then updating every reference to it (matplotlib figure colors, PPTX
   `RGBColor` values, the DOCX tier-color map) throughout the module.
2. **XLSX sheet-name index collision.** The first version's per-outcome sheet-numbering arithmetic
   (`idx`, `idx+2`, `idx+4` for outcome 1 vs. `idx`, `idx+2`, `idx+4` again for outcome 2, with
   `idx` only incrementing by 1 between outcomes) produced sheet tab names that did not sort in
   reading order (`02_FieldRanking_O1, 04_..., 06_..., 03_FieldRanking_O2, 05_..., 07_...`) — a real
   defect a human opening the workbook would immediately notice, caught by directly inspecting
   `wb.sheetnames` after generation rather than only checking the sheet *count*. **Fixed** by
   replacing the index arithmetic with a single incrementing `sheet_no` counter, re-verified to
   produce clean sequential ordering (`02, 03, 04` for outcome 1, `05, 06, 07` for outcome 2).

Separately, this gate's own sandbox verification included a headless-browser render of the
generated dashboard (Playwright + the sandbox's bundled Chromium) with no network egress to
`cdn.plot.ly` available — this is real, valid verification of the **graceful-degradation path**
specifically (every chart box correctly showed the "chart library could not be reached, numbers
are still in the tables" fallback message rather than rendering blank or broken), not a claim that
Plotly itself was verified to render correctly; on the user's own machine, with real network
access, the interactive charts render normally.

## Prerequisite

BP5 Gates 1 through 6 must all have already been real-run by the user (this notebook loads Gate
1's policy, Gate 5's two prioritized root-cause reports, Gate 5's UDAAP check, Gate 4's calibration
curve, and Gate 6's governance summary — all real, already-saved artifact files; it never touches
the Gold layer and never recomputes a Gate 3/4/5 statistic). Per this project's standing
execution-boundary rule, Claude never runs this notebook — only the user does, in the
`home_credit_env` Jupyter kernel. **This is BP5's Gate 7 deliverable, following Gate 6 as this
project's established per-BP closing convention.**

## What this gate does NOT do

- **No re-derivation.** Every number is read from Gate 1's, Gate 4's, or Gate 5's own already-saved
  real artifacts — nothing is recomputed, and the Gold layer is never touched.
- **No causal claims, anywhere.** Every finding remains a statistical ASSOCIATION, never a causal
  claim, throughout every one of the four deliverables.
- **No financial-impact or illustrative-projection content, anywhere** — enforced not just by
  prose but by two of this gate's own structural integrity checks
  (`dashboard_html_no_financial_content`, `xlsx_no_financial_sheet`) and by the manifest's own
  hardcoded `contains_financial_impact_section: False` field.
- **No deployment claim.** This report explicitly frames its recommendation as
  "Decision-Support Use," never "Production" — BP5 has no deployed inference service.

## Real, disclosed design choices in this gate

- The design-system `PALETTE`/`CATEGORICAL_SEQUENCE` constants are reused **verbatim** from
  BP3's/BP4's own real modules (see bug #1 above for how this was actually verified, not merely
  claimed).
- **Both outcomes are always shown side by side**, never behind a toggle — a deliberate choice for
  a 2-outcome report (see "Why this gate looks different" above), kept simple rather than adding
  unneeded UI-state complexity for only two series.
- The category-level findings table shows **all** real category-level findings (46 rows across both
  outcomes) rather than paginating — small enough to display in full, unlike BP4's 37,160-cluster
  table, which genuinely needed pagination.
- Citation/generation timestamps are stamped live at the moment the user real-runs this notebook
  (`datetime.now(timezone.utc)`), never backdated or estimated.

Every real number in this report is read directly from BP5's own real, already real-run-confirmed
Gates 1–6 artifacts. Nothing is estimated, assumed, or synthesized. Every finding remains a
statistical association, never a causal claim, per BP5 Gate 1's own structural disclaimer.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP5 Gate 7 (Executive Rollup Report)

This notebook trains and evaluates nothing itself - every number, table, and chart it produces
was already computed and recorded by Gates 1-6's own real runs, most directly via Gate 5's own
synthesized, citation-backed prioritized root-cause report per outcome. This notebook itself is a
thin orchestrator over src/reporting/bp5_rollup_helpers.py (HYPER) - it does not duplicate that
module's logic.

Zero-fabrication, no assumption-based content, only original notebook output results: every KPI,
table, and chart in every deliverable is read live from Gates 1-6's own already-recorded real
artifacts, or computed live from them by a documented formula over those real values. There is no
illustrative, estimated, or assumption-based content anywhere in this report, and no financial-
impact or illustrative-projection section anywhere in this report - only original notebook output
results are reported, per standing instruction.

If a structural check below fails, it raises AssertionError naming the failing check. Do not
silence it - fix the real underlying issue and re-run.

No execution-based verification of this notebook's own real run was performed by Claude - only
the user runs this notebook for real, in the home_credit_env Jupyter kernel, per this project's
standing execution-boundary rule.
"""

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
from pathlib import Path


def _find_project_root() -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / "PROJECT_STRUCTURE_LOCKED.md").exists():
            return candidate
    cur = Path.cwd()
    for _ in range(8):
        if (cur / "PROJECT_STRUCTURE_LOCKED.md").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth in range(4):
        for candidate in Path.cwd().glob("/".join(["*"] * (depth + 1)) + "/PROJECT_STRUCTURE_LOCKED.md"):
            return candidate.parent
    raise RuntimeError(
        "Could not locate PROJECT_STRUCTURE_LOCKED.md by walking up from or down into the current "
        "working directory. Set C360_PROJECT_ROOT or run this notebook from inside the project."
    )


PROJECT_ROOT = _find_project_root()
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp5_root_cause_driver_analytics" / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp5_root_cause_driver_analytics"
ROLLUP_DIR = REPORTS_DIR / "executive_rollup"
for d in (REPORTS_DIR, ROLLUP_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"[OK] PROJECT_ROOT resolved: {PROJECT_ROOT}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
import sys

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from utils.performance_setup import configure_performance

configure_performance()
print("[OK] WARP performance profile configured.")

# ============================================================
# SECTION 3: Load shared helper module + dependency check
# ============================================================
import importlib

for _pkg in ("docx", "openpyxl", "pptx"):
    try:
        importlib.import_module(_pkg)
    except ImportError as e:
        raise ImportError(
            f"BP5 Gate 7 requires '{_pkg}' (python-docx / openpyxl / python-pptx) to build its "
            f"DOCX/XLSX/PPTX deliverables - not importable: {e}"
        ) from e

from reporting import bp5_rollup_helpers as rollup

BUNDLE = rollup.load_all_gate_artifacts(PROJECT_ROOT)
print("[OK] Loaded real Gate 1-6 artifacts via bp5_rollup_helpers.load_all_gate_artifacts().")

# ============================================================
# SECTION 4: KPI / detail assembly - each builder called exactly once (HYPER)
# ============================================================
KPIS = rollup.build_kpi_bundle(BUNDLE)
GATE1 = rollup.build_gate1_summary(BUNDLE)
GATE6_DETAIL = rollup.build_gate6_governance_detail(BUNDLE)
SUGGESTIONS = rollup.build_smart_suggestions(BUNDLE)

FIELD_DF = {o: rollup.field_ranking_dataframe(BUNDLE, o) for o in rollup.OUTCOME_FIELDS}
CATEGORY_DF = {o: rollup.category_findings_dataframe(BUNDLE, o) for o in rollup.OUTCOME_FIELDS}
SHAP_DF = {o: rollup.shap_findings_dataframe(BUNDLE, o) for o in rollup.OUTCOME_FIELDS}

print(f"[OK] Production recommendation: {KPIS['production_recommendation']['tier']} "
      f"(tier_code={KPIS['production_recommendation']['tier_code']})")
print(f"[OK] Real open items detected: {KPIS['production_recommendation']['n_open_items_detected']}")

# ============================================================
# SECTION 5: Render every static figure once, reused across DOCX and PPTX (HYPER - never
# re-rendered per export format)
# ============================================================
FIGURES: dict[str, bytes] = {}
for _outcome in rollup.OUTCOME_FIELDS:
    _label = rollup.OUTCOME_LABELS[_outcome]
    _snapshot = BUNDLE["gate5_reports"][_outcome]["champion_validation_snapshot"]
    FIGURES[f"cramers_v_{_outcome}"] = rollup.fig_cramers_v_bar(FIELD_DF[_outcome], _label)
    FIGURES[f"shap_{_outcome}"] = rollup.fig_shap_top_features(SHAP_DF[_outcome], _label)
    FIGURES[f"calibration_{_outcome}"] = rollup.fig_calibration_curve(
        BUNDLE["calibration_curve"][_outcome]["calibration_curve"], _label
    )
    FIGURES[f"confusion_{_outcome}"] = rollup.fig_confusion_bar(
        _snapshot["confusion_matrix_at_0_5"], _label
    )
print(f"[OK] Rendered {len(FIGURES)} static figures once.")

# ============================================================
# SECTION 6: Interactive HTML dashboard build - plain string .replace() templating (not Jinja2,
# not f-strings, to avoid brace-escaping conflicts with the template's own CSS/JS), Plotly.js via
# CDN with a `defer` attribute + graceful-degradation fallback if the CDN is unreachable.
# ============================================================
import json
import re
from datetime import datetime, timezone

import numpy as np


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)


_ANSI_ESCAPE_RE = re.compile(r"\x1b\[[0-9;]*m")

_OUTCOME_SHORT = {
    "outcome_1_intervention_required": "O1",
    "outcome_2_timely_response_failure": "O2",
}

_category_findings_flat = []
for _outcome in rollup.OUTCOME_FIELDS:
    for _row in CATEGORY_DF[_outcome].to_dict(orient="records"):
        _row = dict(_row)
        _row["outcome_short"] = _OUTCOME_SHORT[_outcome]
        _category_findings_flat.append(_row)

_kpis_for_dashboard = dict(KPIS)
_kpis_for_dashboard["bootstrap_roc_auc_ci_95_outcome_1_obj"] = {
    "point_estimate": KPIS["held_out_roc_auc_outcome_1"],
    "ci_95_low": KPIS["bootstrap_roc_auc_ci_95_outcome_1"][0],
    "ci_95_high": KPIS["bootstrap_roc_auc_ci_95_outcome_1"][1],
}
_kpis_for_dashboard["bootstrap_roc_auc_ci_95_outcome_2_obj"] = {
    "point_estimate": KPIS["held_out_roc_auc_outcome_2"],
    "ci_95_low": KPIS["bootstrap_roc_auc_ci_95_outcome_2"][0],
    "ci_95_high": KPIS["bootstrap_roc_auc_ci_95_outcome_2"][1],
}

ROLLUP_DATA = {
    "palette": rollup.PALETTE,
    "categorical_sequence": rollup.CATEGORICAL_SEQUENCE,
    "kpis": _kpis_for_dashboard,
    "field_ranking": {
        "outcome_1": FIELD_DF["outcome_1_intervention_required"].to_dict(orient="records"),
        "outcome_2": FIELD_DF["outcome_2_timely_response_failure"].to_dict(orient="records"),
    },
    "shap_findings": {
        "outcome_1": SHAP_DF["outcome_1_intervention_required"].to_dict(orient="records"),
        "outcome_2": SHAP_DF["outcome_2_timely_response_failure"].to_dict(orient="records"),
    },
    "calibration": {
        "outcome_1": BUNDLE["calibration_curve"]["outcome_1_intervention_required"]["calibration_curve"],
        "outcome_2": BUNDLE["calibration_curve"]["outcome_2_timely_response_failure"]["calibration_curve"],
    },
    "category_findings": _category_findings_flat,
    "gate1": GATE1,
    "gate6_detail": GATE6_DETAIL,
    "suggestions": SUGGESTIONS,
}

_rollup_json = json.dumps(ROLLUP_DATA, default=_json_default)
_rollup_json = _rollup_json.replace("</", "<\\/")  # never let the payload close the </script> tag early

_pytest_summary = _ANSI_ESCAPE_RE.sub("", str(GATE6_DETAIL.get("pytest_summary_line", "")))
_syntax_summary = (
    f"{GATE6_DETAIL.get('notebook_syntax_check_n_passed', 0)} passed / "
    f"{GATE6_DETAIL.get('notebook_syntax_check_n_failed', 0)} failed"
)
_generated_at = datetime.now(timezone.utc).isoformat()

_template_path = PROJECT_ROOT / "src" / "reporting" / "templates" / "bp5_dashboard_template.html"
with open(_template_path, "r", encoding="utf-8") as f:
    _html = f.read()
_html = _html.replace("__GENERATED_AT__", _generated_at)
_html = _html.replace("__PYTEST_SUMMARY__", _pytest_summary)
_html = _html.replace("__SYNTAX_SUMMARY__", _syntax_summary)
_html = _html.replace("__ROLLUP_DATA_JSON__", _rollup_json)

_dashboard_path = ROLLUP_DIR / "bp5_executive_rollup_dashboard.html"
with open(_dashboard_path, "w", encoding="utf-8") as f:
    f.write(_html)
print(f"[OK] Wrote interactive HTML dashboard: {_dashboard_path} ({_dashboard_path.stat().st_size:,} bytes)")

# ============================================================
# SECTION 7: DOCX export
# ============================================================
_docx_path = rollup.write_docx_report(
    BUNDLE, KPIS, SUGGESTIONS, FIGURES, ROLLUP_DIR / "bp5_executive_rollup_report.docx"
)
print(f"[OK] Wrote DOCX report: {_docx_path} ({_docx_path.stat().st_size:,} bytes)")

# ============================================================
# SECTION 8: XLSX export
# ============================================================
_xlsx_path = rollup.write_xlsx_workbook(
    BUNDLE, KPIS, SUGGESTIONS, ROLLUP_DIR / "bp5_executive_rollup_workbook.xlsx"
)
print(f"[OK] Wrote XLSX workbook: {_xlsx_path} ({_xlsx_path.stat().st_size:,} bytes)")

# ============================================================
# SECTION 9: PPTX export
# ============================================================
_pptx_path = rollup.write_pptx_deck(
    BUNDLE, KPIS, SUGGESTIONS, FIGURES, ROLLUP_DIR / "bp5_executive_rollup_deck.pptx"
)
print(f"[OK] Wrote PPTX deck: {_pptx_path} ({_pptx_path.stat().st_size:,} bytes)")

# ============================================================
# SECTION 10: Manifest write - flat JSON audit trail, NOT inside the reports/ deliverables folder
# ============================================================
MANIFEST = {
    "bp_id": "bp5",
    "gate": 7,
    "report_name": "BP5 Root-Cause & Driver Analytics - Executive Rollup",
    "outcomes": rollup.OUTCOME_FIELDS,
    "production_recommendation_tier": KPIS["production_recommendation"]["tier"],
    "production_recommendation_tier_code": KPIS["production_recommendation"]["tier_code"],
    "ecoa_reg_b_disparate_impact_applicability": "NOT_APPLICABLE",
    "output_paths": {
        "dashboard_html": str(_dashboard_path.relative_to(PROJECT_ROOT)),
        "report_docx": str(_docx_path.relative_to(PROJECT_ROOT)),
        "workbook_xlsx": str(_xlsx_path.relative_to(PROJECT_ROOT)),
        "deck_pptx": str(_pptx_path.relative_to(PROJECT_ROOT)),
    },
    "output_sizes_bytes": {
        "dashboard_html": _dashboard_path.stat().st_size,
        "report_docx": _docx_path.stat().st_size,
        "workbook_xlsx": _xlsx_path.stat().st_size,
        "deck_pptx": _pptx_path.stat().st_size,
    },
    "contains_financial_impact_section": False,
    "contains_assumption_based_content": False,
    "n_negligible_strength_fields_detected": KPIS["n_negligible_strength_fields_detected"],
    "n_near_zero_precision_outcomes_detected": KPIS["n_near_zero_precision_outcomes_detected"],
    "association_not_causation_disclaimer_carried_forward": True,
    "generated_at_utc": _generated_at,
}
_manifest_path = ARTIFACTS_DIR / "executive_rollup_manifest.json"
with open(_manifest_path, "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2, default=_json_default)
print(f"[OK] Wrote manifest: {_manifest_path}")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass.
# ============================================================
from docx import Document as _ReopenDocument
from openpyxl import load_workbook as _reopen_workbook
from pptx import Presentation as _ReopenPresentation

_reopened_docx = _ReopenDocument(str(_docx_path))
_reopened_xlsx = _reopen_workbook(str(_xlsx_path))
_reopened_pptx = _ReopenPresentation(str(_pptx_path))

with open(_dashboard_path, "r", encoding="utf-8") as f:
    _saved_html = f.read()
_match = re.search(
    r'<script id="rollup-data" type="application/json">(.*?)</script>', _saved_html, re.DOTALL
)
assert _match is not None, "Could not find the embedded rollup-data <script> tag in the saved HTML."
_reparsed_rollup_data = json.loads(_match.group(1))

checks: dict[str, bool] = {
    "dashboard_html_nonempty": _dashboard_path.stat().st_size > 10_000,
    "docx_nonempty": _docx_path.stat().st_size > 10_000,
    "xlsx_nonempty": _xlsx_path.stat().st_size > 5_000,
    "pptx_nonempty": _pptx_path.stat().st_size > 10_000,
    "docx_reopens_and_has_tables": len(_reopened_docx.tables) >= 3,
    "xlsx_reopens_and_has_expected_sheet_count": len(_reopened_xlsx.sheetnames) == 10,
    "xlsx_has_readme_sheet": "00_ReadMe" in _reopened_xlsx.sheetnames,
    "xlsx_no_financial_sheet": not any("Financial" in name for name in _reopened_xlsx.sheetnames),
    "pptx_reopens_and_has_expected_slide_count": len(_reopened_pptx.slides._sldIdLst) == 14,
    "dashboard_html_no_financial_content": (
        "financial" not in _rollup_json.lower() and "Financial Impact" not in _saved_html
    ),
    "dashboard_html_no_unresolved_placeholders": "__" not in _saved_html.replace("__init__", ""),
    "dashboard_embedded_json_reparses_cleanly": isinstance(_reparsed_rollup_data, dict),
    "dashboard_field_ranking_row_counts_match_source": (
        len(_reparsed_rollup_data["field_ranking"]["outcome_1"]) == len(FIELD_DF["outcome_1_intervention_required"])
        and len(_reparsed_rollup_data["field_ranking"]["outcome_2"]) == len(FIELD_DF["outcome_2_timely_response_failure"])
    ),
    "dashboard_category_findings_row_count_matches_source": (
        len(_reparsed_rollup_data["category_findings"]) == len(_category_findings_flat)
    ),
    "production_recommendation_tier_code_valid": KPIS["production_recommendation"]["tier_code"] in (1, 2, 3),
    "ecoa_never_triggers_this_bps_tier": (
        KPIS["production_recommendation"]["ecoa_reg_b_disparate_impact_applicability"] == "NOT_APPLICABLE"
    ),
    "outputs_written_under_executive_rollup_folder": all(
        str(p).startswith(str(ROLLUP_DIR)) for p in (_dashboard_path, _docx_path, _xlsx_path, _pptx_path)
    ),
    "manifest_declares_zero_financial_content": (
        MANIFEST["contains_financial_impact_section"] is False
        and MANIFEST["contains_assumption_based_content"] is False
    ),
}

_failed = [name for name, ok in checks.items() if not ok]
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert not _failed, f"BP5 Gate 7 structural integrity checks failed: {_failed}"

print(
    "\n[ALL CHECKS PASSED] BP5 Gate 7 (Executive Rollup Report) complete. "
    f"Recommendation: {KPIS['production_recommendation']['tier']}. "
    "This is BP5's Gate 7 deliverable - the audience-facing synthesis of Gates 1-6's own real work."
)
